In [ ]:
# Cell 1: Imports and settings
from __future__ import annotations

import importlib.util
import json
import math
import os
import re
import subprocess
import sys
import time
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
}
missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    )

import numpy as np
import pandas as pd
from scipy import signal
from scipy.io import loadmat
from google.colab import drive

EEG_SAMPLE_RATE = 200.0
NIRS_SAMPLE_RATE = 10.0
EPOCH_START_SEC = -5.0
EPOCH_END_SEC = 25.0
BASELINE_START_SEC = -5.0
BASELINE_END_SEC = -2.0
TASK_END_SEC = 12.0
EEG_FILTER_ORDER = 6
EEG_BP = (1.0, 40.0)
NIRS_FILTER_ORDER = 3
NIRS_LP_HZ = 0.2
PROJECT_ROOT = Path(
    "/content/gdrive/MyDrive/HonorProject/HemoBoundary/SHIN"
)

EXPECTED_TRIALS = 60
EXPECTED_TRIALS_PER_SESSION = 20
EXPECTED_WG_PER_SESSION = 10
EXPECTED_BL_PER_SESSION = 10

EPOCH_TMIN = EPOCH_START_SEC
EPOCH_TMAX = EPOCH_END_SEC
BASELINE = (BASELINE_START_SEC, BASELINE_END_SEC)


In [ ]:
# Cell 2
def mount_google_drive() -> None:
    if Path('/content/gdrive/MyDrive').exists():
        return
    try:
        drive.mount('/content/gdrive', force_remount=False)
    except Exception as exc:
        raise RuntimeError('Google Drive could not be mounted at /content/gdrive.') from exc

def get_project_root() -> Path:
    mount_google_drive()
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    return PROJECT_ROOT

def to_string_array(vals: Sequence[Any]) -> np.ndarray:
    return np.asarray([str(v) for v in vals], dtype=str)

def write_json_atomic(obj: object, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding='utf-8')
    tmp.replace(path)

def load_mat_file(path: Path) -> Dict[str, Any]:
    try:
        return loadmat(path, squeeze_me=True, struct_as_record=False)
    except NotImplementedError as exc:
        raise RuntimeError(f'{path} is MATLAB v7.3/HDF5; this dataset is expected to use classic MAT files.') from exc

def get_mat_variable(path: Path, expected: str) -> Any:
    d = {k: v for k, v in load_mat_file(path).items() if not k.startswith('__')}
    if expected in d:
        return d[expected]
    if len(d) == 1:
        return next(iter(d.values()))
    for k, v in d.items():
        if k.lower() == expected.lower() or expected.lower() in k.lower():
            return v
    raise KeyError(f'{path.name}: cannot find {expected}; variables={list(d)}')

def get_mat_field(obj: Any, name: str, default: Any=None) -> Any:
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(name, default)
    if hasattr(obj, name):
        return getattr(obj, name)
    if isinstance(obj, np.void) and obj.dtype.names and (name in obj.dtype.names):
        return obj[name]
    return default


In [ ]:
# Cell 3
def extract_strings(x: Any) -> List[str]:
    if x is None:
        return []
    a = np.asarray(x, dtype=object).reshape(-1)
    out = []
    for v in a:
        if isinstance(v, bytes):
            out.append(v.decode(errors='replace'))
        elif isinstance(v, np.ndarray) and v.ndim == 0:
            out.append(str(v.item()))
        else:
            out.append(str(v))
    return out

def extract_continuous_signal(cnt: Any) -> Tuple[np.ndarray, float, List[str]]:
    x = np.asarray(get_mat_field(cnt, 'x'), dtype=float)
    fs = float(np.asarray(get_mat_field(cnt, 'fs')).reshape(-1)[0])
    clab = extract_strings(get_mat_field(cnt, 'clab'))
    if x.ndim != 2:
        raise RuntimeError(f'Expected continuous x [time,channel], got {x.shape}')
    if clab and x.shape[1] != len(clab) and (x.shape[0] == len(clab)):
        x = x.T
    if clab and x.shape[1] != len(clab):
        raise RuntimeError(f'Channel-label mismatch x={x.shape}, clab={len(clab)}')
    return (x, fs, clab)

def extract_nirs_concentrations(cnt: Any) -> Tuple[np.ndarray, np.ndarray, float, List[str]]:
    oxy = get_mat_field(cnt, 'oxy')
    deoxy = get_mat_field(cnt, 'deoxy')
    if oxy is None or deoxy is None:
        raise RuntimeError('NIRS cnt_wg must contain .oxy and .deoxy structures')
    xo, fso, co = extract_continuous_signal(oxy)
    xd, fsd, cd = extract_continuous_signal(deoxy)
    if xo.shape != xd.shape or abs(fso - fsd) > 1e-09:
        raise RuntimeError(f'HbO/HbR mismatch: {xo.shape}/{xd.shape}, {fso}/{fsd}')
    clab = co if co else cd
    if co and cd and (co != cd):
        warnings.warn('HbO/HbR channel labels differ; using HbO labels')
    return (xo, xd, fso, clab)

def decode_event_markers(mrk: Any, modality: str) -> pd.DataFrame:
    times_ms = np.asarray(get_mat_field(mrk, 'time'), dtype=float).reshape(-1)
    y = np.asarray(get_mat_field(mrk, 'y'), dtype=float)
    class_names = [s.strip() for s in extract_strings(get_mat_field(mrk, 'className'))]
    n = len(times_ms)
    if y.ndim == 1:
        codes = y.reshape(-1)
        if len(codes) != n:
            raise RuntimeError(f'Marker y length {len(codes)} != times {n}')
        if modality == 'EEG':
            label = np.where(codes == 16, 'WG', np.where(codes == 32, 'BL', 'UNKNOWN'))
        else:
            label = np.where(codes == 1, 'WG', np.where(codes == 2, 'BL', 'UNKNOWN'))
    else:
        if y.shape[-1] == n:
            Y = y.reshape((-1, n))
        elif y.shape[0] == n:
            Y = y.reshape((n, -1)).T
        else:
            raise RuntimeError(f'Cannot align mrk.y shape {y.shape} to {n} markers')
        idx = np.argmax(Y, axis=0)
        if class_names and len(class_names) >= Y.shape[0]:
            normalized = []
            for c in class_names[:Y.shape[0]]:
                cu = c.upper()
                if 'WG' in cu or 'VERBAL' in cu or 'FLU' in cu:
                    normalized.append('WG')
                elif 'BL' in cu or 'BASE' in cu:
                    normalized.append('BL')
                else:
                    normalized.append(c)
            label = np.asarray([normalized[i] for i in idx], dtype=str)
        else:
            label = np.where(idx == 0, 'WG', 'BL')
    frame = pd.DataFrame({'marker_index': np.arange(n), 'onset_sec': times_ms / 1000.0, 'label': label})
    frame = frame[frame['label'].isin(['WG', 'BL'])].sort_values('onset_sec', kind='stable').reset_index(drop=True)
    if len(frame) != EXPECTED_TRIALS:
        raise RuntimeError(f'{modality}: expected {EXPECTED_TRIALS} WG/BL markers, found {len(frame)}')
    frame['trial_global'] = np.arange(EXPECTED_TRIALS, dtype=int)
    frame['session'] = frame['trial_global'] // EXPECTED_TRIALS_PER_SESSION + 1
    frame['trial_in_session'] = frame['trial_global'] % EXPECTED_TRIALS_PER_SESSION
    for session_id, g in frame.groupby('session'):
        counts = g['label'].value_counts().to_dict()
        if counts.get('WG', 0) != EXPECTED_WG_PER_SESSION or counts.get('BL', 0) != EXPECTED_BL_PER_SESSION:
            raise RuntimeError(f'{modality} session {session_id}: expected 10 WG/10 BL, got {counts}')
    return frame


In [ ]:
# Cell 4
def remove_eog_regression(x: np.ndarray, n_scalp: int) -> Tuple[np.ndarray, Dict[str, object]]:
    scalp = np.asarray(x[:, :n_scalp], float)
    eog = np.asarray(x[:, n_scalp:], float)
    if eog.shape[1] == 0:
        return (scalp.copy(), {'eog_channels': 0, 'regression_applied': 0, 'variance_removed_fraction': 0.0})
    finite = np.isfinite(eog).all(axis=1) & np.isfinite(scalp).all(axis=1)
    if finite.sum() < 100:
        return (scalp.copy(), {'eog_channels': eog.shape[1], 'regression_applied': 0, 'variance_removed_fraction': np.nan})
    E = eog.copy()
    E -= np.nanmean(E[finite], axis=0, keepdims=True)
    S = scalp.copy()
    mu = np.nanmean(S[finite], axis=0, keepdims=True)
    Sd = S - mu
    beta = np.linalg.pinv(E[finite]) @ Sd[finite]
    cleaned = Sd - E @ beta + mu
    before = np.nanvar(S[finite])
    after = np.nanvar(cleaned[finite])
    frac = 1.0 - after / before if before > 0 else np.nan
    return (cleaned, {'eog_channels': eog.shape[1], 'regression_applied': 1, 'variance_removed_fraction': float(frac)})

def apply_forward_filter(x: np.ndarray, sos: np.ndarray) -> np.ndarray:
    return signal.sosfilt(sos, x, axis=0)

def apply_zero_phase_filter(x: np.ndarray, sos: np.ndarray) -> np.ndarray:
    return signal.sosfiltfilt(sos, x, axis=0)


In [ ]:
# Cell 5
def extract_epochs(x: np.ndarray, fs: float, onsets: Sequence[float]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    n_samp = int(round((EPOCH_TMAX - EPOCH_TMIN) * fs)) + 1
    times = np.arange(n_samp, dtype=float) / fs + EPOCH_TMIN
    eps = np.full((len(onsets), x.shape[1], n_samp), np.nan, dtype=np.float32)
    ok = np.zeros(len(onsets), dtype=bool)
    for i, onset in enumerate(onsets):
        center = int(round(float(onset) * fs))
        start = center + int(round(EPOCH_TMIN * fs))
        stop = start + n_samp
        if start < 0 or stop > x.shape[0]:
            continue
        seg = x[start:stop, :].T
        if seg.shape[1] == n_samp:
            eps[i] = seg.astype(np.float32)
            ok[i] = np.isfinite(seg).all()
    return (eps, times, ok)

def baseline_correct(ep: np.ndarray, times: np.ndarray) -> np.ndarray:
    mask = (times >= BASELINE[0] - 1e-09) & (times <= BASELINE[1] + 1e-09)
    if mask.sum() < 2:
        raise RuntimeError('Baseline mask too small')
    base = np.nanmean(ep[:, :, mask], axis=2, keepdims=True)
    return (ep - base).astype(np.float32)

def robust_z_score(v: np.ndarray) -> np.ndarray:
    a = np.asarray(v, float)
    med = np.nanmedian(a)
    mad = np.nanmedian(np.abs(a - med))
    scale = 1.4826 * mad
    if not np.isfinite(scale) or scale < 1e-12:
        scale = np.nanstd(a)
    if not np.isfinite(scale) or scale < 1e-12:
        return np.zeros_like(a)
    return (a - med) / scale

def compute_trial_quality(eeg: np.ndarray, eeg_times: np.ndarray, hbo: np.ndarray, hbr: np.ndarray, nirs_times: np.ndarray, eeg_fs: float, nirs_fs: float) -> pd.DataFrame:
    em = (eeg_times >= 0) & (eeg_times <= TASK_END_SEC)
    nm = (nirs_times >= 0) & (nirs_times <= TASK_END_SEC)
    eeg_abs = np.nanmax(np.abs(eeg[:, :, em]), axis=(1, 2))
    eeg_ptp = np.nanmax(np.ptp(eeg[:, :, em], axis=2), axis=1)
    dhbo = np.diff(hbo[:, :, nm], axis=2) * nirs_fs
    dhbr = np.diff(hbr[:, :, nm], axis=2) * nirs_fs
    nirs_deriv = np.nanmax(np.maximum(np.abs(dhbo), np.abs(dhbr)), axis=(1, 2))
    out = pd.DataFrame({'eeg_max_abs': eeg_abs, 'eeg_max_ptp': eeg_ptp, 'nirs_max_abs_derivative': nirs_deriv})
    out['eeg_abs_robust_z'] = robust_z_score(eeg_abs)
    out['eeg_ptp_robust_z'] = robust_z_score(eeg_ptp)
    out['nirs_deriv_robust_z'] = robust_z_score(nirs_deriv)
    out['artifact_review_flag'] = ((np.abs(out['eeg_abs_robust_z']) > 6) | (np.abs(out['eeg_ptp_robust_z']) > 6) | (np.abs(out['nirs_deriv_robust_z']) > 6)).astype(int)
    return out


In [ ]:
# Cell 6
def discover_participant_directories(raw: Path) -> List[Tuple[str, Path, Path]]:
    rows = []
    for eeg_dir in sorted(raw.glob('VP*-EEG')):
        m = re.search('VP(\\d{3})-EEG', eeg_dir.name)
        if not m:
            continue
        vp = f'VP{int(m.group(1)):03d}'
        nirs_dir = raw / f'{vp}-NIRS'
        if nirs_dir.is_dir():
            rows.append((vp, eeg_dir, nirs_dir))
    return rows

def run_validation_checks() -> None:

    class Dummy:
        pass
    m = Dummy()
    m.time = np.arange(60) * 25000.0
    m.y = np.zeros((2, 60))
    for s in range(3):
        ids = np.arange(s * 20, (s + 1) * 20)
        m.y[0, ids[:10]] = 1
        m.y[1, ids[10:]] = 1
    m.className = np.array(['WG', 'BL'], dtype=object)
    f = decode_event_markers(m, 'EEG')
    assert len(f) == 60 and f.groupby('session')['label'].value_counts().min() == 10
    x = np.arange(200 * 100 * 2, dtype=float).reshape(200 * 100, 2)
    ep, t, ok = extract_epochs(x, 200, [10, 40, 70])
    assert ep.shape[1] == 2 and ok.all()
    print('SHIN preprocessing checks: PASS')


In [ ]:
# Cell 7
def run_preprocessing() -> None:
    print('=' * 136)
    print('SHIN 02 — PREPROCESS WORD-GENERATION EEG–NIRS')
    print('=' * 136)
    run_validation_checks()
    root = get_project_root()
    raw = root / '01_download' / 'raw_matlab'
    stage = root / '02_preprocessed'
    audit = stage / 'audits'
    stage.mkdir(parents=True, exist_ok=True)
    audit.mkdir(parents=True, exist_ok=True)
    if not raw.is_dir():
        raise FileNotFoundError(f'Input directory not found: {raw}')
    subjects = discover_participant_directories(raw)
    if not subjects:
        raise FileNotFoundError(f'No VPxxx-EEG/NIRS pairs found in {raw}')
    alignment_all = []
    eeg_audit = []
    nirs_audit = []
    quality_all = []
    summary = []
    failures = []
    for idx, (vp, eeg_dir, nirs_dir) in enumerate(subjects, 1):
        print(f'\n[{idx:02d}/{len(subjects):02d}] {vp}')
        try:
            eeg_cnt = get_mat_variable(eeg_dir / 'cnt_wg.mat', 'cnt_wg')
            eeg_mrk = get_mat_variable(eeg_dir / 'mrk_wg.mat', 'mrk_wg')
            eeg_mnt = get_mat_variable(eeg_dir / 'mnt_wg.mat', 'mnt_wg')
            nirs_cnt = get_mat_variable(nirs_dir / 'cnt_wg.mat', 'cnt_wg')
            nirs_mrk = get_mat_variable(nirs_dir / 'mrk_wg.mat', 'mrk_wg')
            nirs_mnt = get_mat_variable(nirs_dir / 'mnt_wg.mat', 'mnt_wg')
            eeg_x, eeg_fs, eeg_clab = extract_continuous_signal(eeg_cnt)
            mnt_eeg_clab = extract_strings(get_mat_field(eeg_mnt, 'clab'))
            if not eeg_clab:
                eeg_clab = mnt_eeg_clab
            if len(eeg_clab) < 28:
                raise RuntimeError(f'{vp}: expected >=28 EEG/EOG labels, got {len(eeg_clab)}')
            n_scalp = len(mnt_eeg_clab) - 2 if len(mnt_eeg_clab) >= 30 else min(28, len(eeg_clab) - 2)
            if n_scalp != 28:
                warnings.warn(f'{vp}: inferred scalp count {n_scalp}; expected 28')
            scalp_names = eeg_clab[:n_scalp]
            eog_names = eeg_clab[n_scalp:]
            eeg_mark = decode_event_markers(eeg_mrk, 'EEG')
            nirs_mark = decode_event_markers(nirs_mrk, 'NIRS')
            if eeg_mark['label'].tolist() != nirs_mark['label'].tolist():
                raise RuntimeError(f'{vp}: EEG/NIRS WG/BL marker sequence mismatch')
            meta = eeg_mark[['trial_global', 'session', 'trial_in_session', 'label']].copy()
            meta['participant'] = vp
            meta['trial_uid'] = [f'{vp}__S{int(s):02d}__T{int(t):02d}' for s, t in zip(meta.session, meta.trial_in_session)]
            meta['eeg_onset_sec'] = eeg_mark['onset_sec'].to_numpy(float)
            meta['nirs_onset_sec'] = nirs_mark['onset_sec'].to_numpy(float)
            meta['task_end_sec_relative'] = TASK_END_SEC
            eg = np.r_[np.diff(meta['eeg_onset_sec']), np.nan]
            ng = np.r_[np.diff(meta['nirs_onset_sec']), np.nan]
            meta['eeg_next_task_gap_sec'] = eg
            meta['nirs_next_task_gap_sec'] = ng
            eeg_reg, eog_info = remove_eog_regression(eeg_x, n_scalp)
            if not 150 <= eeg_fs <= 250:
                warnings.warn(f'{vp}: EEG fs={eeg_fs}, expected ~200 Hz MATLAB resource')
            eeg_sos = signal.butter(EEG_FILTER_ORDER, EEG_BP, btype='bandpass', fs=eeg_fs, output='sos')
            eeg_causal = apply_forward_filter(eeg_reg, eeg_sos)
            eeg_zero = apply_zero_phase_filter(eeg_reg, eeg_sos)
            ep_ec, t_e, ok_ec = extract_epochs(eeg_causal, eeg_fs, meta.eeg_onset_sec)
            ep_ez, _, ok_ez = extract_epochs(eeg_zero, eeg_fs, meta.eeg_onset_sec)
            ep_ec = baseline_correct(ep_ec, t_e)
            ep_ez = baseline_correct(ep_ez, t_e)
            hbo, hbr, nirs_fs, nirs_clab = extract_nirs_concentrations(nirs_cnt)
            if not nirs_clab:
                nirs_clab = extract_strings(get_mat_field(nirs_mnt, 'clab'))
            if hbo.shape[1] != 36:
                warnings.warn(f'{vp}: NIRS channels={hbo.shape[1]}, expected channel count is 36')
            ep_hbo_raw, t_n, ok_no = extract_epochs(hbo, nirs_fs, meta.nirs_onset_sec)
            ep_hbr_raw, _, ok_nd = extract_epochs(hbr, nirs_fs, meta.nirs_onset_sec)
            ep_hbo_raw = baseline_correct(ep_hbo_raw, t_n)
            ep_hbr_raw = baseline_correct(ep_hbr_raw, t_n)
            n_sos = signal.butter(NIRS_FILTER_ORDER, NIRS_LP_HZ, btype='lowpass', fs=nirs_fs, output='sos')
            hbo_c = apply_forward_filter(hbo, n_sos)
            hbr_c = apply_forward_filter(hbr, n_sos)
            hbo_z = apply_zero_phase_filter(hbo, n_sos)
            hbr_z = apply_zero_phase_filter(hbr, n_sos)
            ep_hbo_c, _, ok_nc = extract_epochs(hbo_c, nirs_fs, meta.nirs_onset_sec)
            ep_hbr_c, _, ok_ncr = extract_epochs(hbr_c, nirs_fs, meta.nirs_onset_sec)
            ep_hbo_z, _, ok_nz = extract_epochs(hbo_z, nirs_fs, meta.nirs_onset_sec)
            ep_hbr_z, _, ok_nzr = extract_epochs(hbr_z, nirs_fs, meta.nirs_onset_sec)
            for arr in (ep_hbo_c, ep_hbr_c, ep_hbo_z, ep_hbr_z):
                arr[:] = baseline_correct(arr, t_n)
            all_ok = ok_ec & ok_ez & ok_no & ok_nd & ok_nc & ok_ncr & ok_nz & ok_nzr
            if all_ok.sum() < 50:
                raise RuntimeError(f'{vp}: only {all_ok.sum()}/60 trials have complete multimodal epochs')
            quality = compute_trial_quality(ep_ec, t_e, ep_hbo_raw, ep_hbr_raw, t_n, eeg_fs, nirs_fs)
            quality.insert(0, 'trial_uid', meta.trial_uid)
            quality.insert(0, 'participant', vp)
            quality['complete_multimodal_epoch'] = all_ok.astype(int)
            quality_all.append(quality)
            meta['complete_multimodal_epoch'] = all_ok.astype(int)
            align = meta.copy()
            alignment_all.append(align)
            np.savez_compressed(stage / f'{vp}_WG_ALIGNED_PREPROCESSED.npz', eeg_causal_epochs=ep_ec.astype(np.float32), eeg_zero_phase_epochs=ep_ez.astype(np.float32), eeg_times=t_e.astype(np.float64), eeg_sfreq=np.asarray([eeg_fs], np.float64), eeg_channel_names=to_string_array(scalp_names), eeg_eog_names=to_string_array(eog_names), hbo_raw_epochs=ep_hbo_raw.astype(np.float32), hbr_raw_epochs=ep_hbr_raw.astype(np.float32), hbo_causal_lp02_epochs=ep_hbo_c.astype(np.float32), hbr_causal_lp02_epochs=ep_hbr_c.astype(np.float32), hbo_zero_lp02_epochs=ep_hbo_z.astype(np.float32), hbr_zero_lp02_epochs=ep_hbr_z.astype(np.float32), nirs_times=t_n.astype(np.float64), nirs_sfreq=np.asarray([nirs_fs], np.float64), nirs_channel_names=to_string_array(nirs_clab if nirs_clab else [f'NIRS{i + 1:02d}' for i in range(hbo.shape[1])]), participant=to_string_array([vp] * len(meta)), trial_uid=to_string_array(meta.trial_uid), session=meta.session.to_numpy(int), trial_in_session=meta.trial_in_session.to_numpy(int), trial_global=meta.trial_global.to_numpy(int), label=to_string_array(meta.label), y=(meta.label.to_numpy(str) == 'WG').astype(int), eeg_onset_sec=meta.eeg_onset_sec.to_numpy(float), nirs_onset_sec=meta.nirs_onset_sec.to_numpy(float), task_end_sec=np.full(len(meta), TASK_END_SEC, dtype=float), complete_multimodal_epoch=all_ok.astype(np.uint8), artifact_review_flag=quality.artifact_review_flag.to_numpy(np.uint8))
            eeg_audit.append({'participant': vp, 'eeg_fs': eeg_fs, 'continuous_samples': len(eeg_x), 'continuous_channels': eeg_x.shape[1], 'scalp_channels': n_scalp, 'eog_channels': len(eog_names), 'eog_regression_applied': eog_info['regression_applied'], 'variance_removed_fraction': eog_info['variance_removed_fraction'], 'primary_filter': '6th-order forward Butterworth 1-40 Hz', 'sensitivity_filter': '6th-order zero-phase Butterworth 1-40 Hz', 'baseline': '-5 to -2 s', 'epoch': '-5 to +25 s'})
            nirs_audit.append({'participant': vp, 'nirs_fs': nirs_fs, 'continuous_samples': len(hbo), 'channels': hbo.shape[1], 'primary_branch': 'provided HbO/HbR, no temporal filter', 'causal_sensitivity': '3rd-order forward Butterworth low-pass 0.2 Hz', 'zero_phase_sensitivity': '3rd-order zero-phase Butterworth low-pass 0.2 Hz', 'baseline': '-5 to -2 s', 'epoch': '-5 to +25 s', 'SCI_available': 0})
            summary.append({'participant': vp, 'trials': len(meta), 'WG': int((meta.label == 'WG').sum()), 'BL': int((meta.label == 'BL').sum()), 'complete_epochs': int(all_ok.sum()), 'artifact_review_flags': int(quality.artifact_review_flag.sum())})
            print(f'  saved trials={len(meta)} complete={all_ok.sum()} review_flags={quality.artifact_review_flag.sum()}')
        except Exception as exc:
            failures.append({'participant': vp, 'error_type': type(exc).__name__, 'error': repr(exc)})
            print('  Error:', repr(exc))
    if alignment_all:
        pd.concat(alignment_all, ignore_index=True).to_csv(audit / 'trial_alignment.csv', index=False)
    pd.DataFrame(summary).to_csv(audit / 'participant_summary.csv', index=False)
    pd.DataFrame(eeg_audit).to_csv(audit / 'eeg_preprocessing.csv', index=False)
    pd.DataFrame(nirs_audit).to_csv(audit / 'nirs_preprocessing.csv', index=False)
    if quality_all:
        pd.concat(quality_all, ignore_index=True).to_csv(audit / 'trial_quality_review.csv', index=False)
    pd.DataFrame(failures, columns=['participant', 'error_type', 'error']).to_csv(audit / 'failures.csv', index=False)
    write_json_atomic({'completed_at_unix': time.time()}, audit / 'completion.json')
    print('\n' + '=' * 136)
    print('SHIN preprocessing finished')
    print('=' * 136)
    print('Succeeded:', len(summary), 'Failed:', len(failures))
    print('Output:', stage)
    print('Preprocessing outputs saved.')
if __name__ == '__main__':
    run_preprocessing()
